# Clean ECG signal images on EC2 — HEEDB clean12

This standalone notebook reads the physician-verified **HEEDB clean12** selection (`ECG_500Hz_clean12.tar.gz`: 113,358 native-500 Hz records, 12 conditions, unsplit) and renders **one clean image per ECG record** with the real ECG-image-kit (vendored `ecg_image_v1_code.tar.gz`, dataset_generation_code PLAN §6). The rendering is identical to the kit-v1 notebook (`../clean_ecg_signal_images_ec2/`); only the source, the record discovery, and the output locations change. It does not filter, resample, normalize, synthesize, or augment waveform samples. Each record is staged to a canonical scratch WFDB `.dat` (fmt 16, gain 1000, canonical lead order, int16 round-trip verified — this dodges the kit's `.mat` gain bug, PLAN §0.5) and rendered by the kit's `gen_ecg_images_from_data_batch.py` with the reference flags (`--num_columns 4 --full_mode II -r 200 --pad_inches 0 --calibration_pulse 1 --random_bw 1`, `ECG_ROW_GAP=4.4 ECG_RHYTHM_GAP=5.5 ECG_BOTTOM_RESERVE=1.9`, fixed `Arial.ttf`): standard clinical 3x4 layout (2.5 s per lead panel) plus a continuous full 10 s Lead II rhythm strip at 25 mm/s, 10 mm/mV, with calibration pulses and lead names and no printed ID header. The grey 2200x1700 master is recoloured to the canonical clean red grid (fixed LUT, PLAN §10) and resized with LANCZOS to 994x768 plus 15 px paper-colour pads on each side, giving a final 1024x768 PNG. Pixel rasterization plus the fixed recolour/resize is the only lossy step.

Run cells in order on Linux. Copy **this notebook, `ecg_image_v1_code.tar.gz`, and `ecg_image_v1_code.tar.gz.sha256`** (all three are in this folder) to the same folder on EC2, then launch Jupyter from that folder. The notebook verifies the local kit bundle against its SHA-256 sidecar; if no local copy is found, it tries `S3_KIT_SOURCE` (§1) when those two kit objects have been uploaded there. It downloads the source archive and its `.sha256` sidecar from `s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/`, verifies the complete archive before extraction, verifies the kit bundle against its sidecar, refuses to render unless every `patch_kit` marker is present, and writes its own worker module into the configured working directory. The output mirrors the source tree (`I0001/WFDB/...`, `I0006/WFDB/...`) under `images/`. The package includes images, the source record manifest (labels, patient keys, sha256), the selection report and documentation, a rendering manifest, a failure log, a configuration snapshot, and a summary. It is uploaded to `s3://cardiosentry/dataset/` with a `.sha256` sidecar. Reruns reuse valid existing images when rendering settings match.

**Expected source layout:** `ECG_500Hz_clean12/manifest.csv` (one row per record; `record_path` is relative to the folder) and the records it lists under `I0001/WFDB/.../*.hea` (`.mat`, fmt 16+24) and `I0006/WFDB/.../*.hea` (`.dat`, fmt 16). Every record has 12 leads, 5,000 samples at native 500 Hz. Records are discovered from `manifest.csv`, never by globbing, so stray headers can never be rendered. There is no train/validation/test split in this dataset; split by `patient_key` from `record_manifest.csv` at training time.


In [ ]:
# 1. Configuration: edit before running any later cell.
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path

S3_SIGNAL_PREFIX = "s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/"
S3_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz"
S3_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz.sha256"
# Fallback only: the kit ships next to this notebook. Upload both objects here
# if you would rather not copy them to the instance.
S3_KIT_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz"
S3_KIT_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz.sha256"
WORK_DIR = Path.home() / "ecg_clean12_render"
LOCAL_ARCHIVE = WORK_DIR / "input" / "ECG_500Hz_clean12.tar.gz"
INPUT_DIR = WORK_DIR / "input" / "ECG_500Hz_clean12"
OUTPUT_DIR = WORK_DIR / "clean_ecg_signal_images_clean12"
S3_UPLOAD_PREFIX = "s3://cardiosentry/dataset/"
ARCHIVE_FORMAT = "tar"            # "tar" (fast; PNG is compressed) or "tar.gz"
ARCHIVE_BASENAME = "clean_ecg_signal_images_clean12"

# clean12 source contract (dataset-files/HEEDB/clean12_selection_2026-10-01):
# every record is native 500 Hz, 12 leads, 5,000 samples. §5 refuses any
# other count or dimension instead of silently rendering a different set.
EXPECTED_RECORDS = 113_358
EXPECTED_FS = 500
EXPECTED_SAMPLES = 5000
CONDITIONS = ["SB", "STACH", "LBBB", "LQT", "AF", "AFL",
              "3AVB", "2AVB", "1AVB", "STEMI", "STE", "NORM"]

# Each worker runs one single-threaded kit (matplotlib) process at a time and
# the parent only collects results, so every vCPU gets a worker. c8a has no
# SMT (each vCPU is a physical core): 16 workers on c8a.4xlarge.
WORKERS = max(1, os.cpu_count() or 1)
BATCH_SIZE = 48                    # records per kit invocation; amortizes the 1-3 s kit import
MAX_PENDING_BATCHES = 2 * WORKERS  # bounds task/result memory
PROGRESS_EVERY = 1000
# Kit scratch (staged .dat + grey masters, deleted after every batch) lives in
# RAM when /dev/shm is writable, so per-batch render I/O never touches EBS.
KIT_SCRATCH_DIR = (Path("/dev/shm") / "ecg_clean12_kit_scratch"
                   if os.access("/dev/shm", os.W_OK) else WORK_DIR / "kit_scratch")
# S3 transfers: c8a.4xlarge has up to 12.5 Gbps; 8 threads leave most of it idle.
S3_TRANSFER = dict(max_concurrency=32, multipart_chunksize=64 * 2**20, use_threads=True)

# Rendering uses the real ECG-image-kit (PLAN §6): 200-DPI grey masters
# (2200x1700 px), 3x4 (2.5 s/lead) + full 10 s Lead II rhythm strip,
# 25 mm/s, 10 mm/mV, calibration pulses, fixed Arial; recoloured to the
# canonical clean red grid and resized to a final 1024x768 PNG (PLAN §7.3).
IMAGE_FORMAT = "PNG"
FINAL_IMAGE_WIDTH = 1024
FINAL_IMAGE_HEIGHT = 768
PNG_COMPRESS_LEVEL = 3             # fast lossless PNG
MAX_DURATION_S = 10.0
KIT_SEED = 20260918                # kit -se (dead code in the kit; passed anyway)
KIT_FIXED_FONT = "Arial.ttf"       # clean images, every split (PLAN §6.7)
KIT_RENDER_FLAGS = ["--num_columns", "4", "--full_mode", "II", "-r", "200",
                    "--pad_inches", "0", "--calibration_pulse", "1",
                    "--random_bw", "1"]  # grey master; recoloured clean-red after
KIT_ENV = {"ECG_ROW_GAP": "4.4", "ECG_RHYTHM_GAP": "5.5",
           "ECG_BOTTOM_RESERVE": "1.9"}
RENDER_VERSION = 4  # 4 = real ECG-image-kit + clean-red recolour + 1024x768

assert IMAGE_FORMAT == "PNG", "This implementation produces lossless PNG files"
assert ARCHIVE_FORMAT in {"tar", "tar.gz"}
assert WORKERS >= 1 and BATCH_SIZE >= 1 and MAX_PENDING_BATCHES >= WORKERS
assert (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT) == (1024, 768)
assert 1 <= PNG_COMPRESS_LEVEL <= 9 and MAX_DURATION_S > 0
print("Working directory:", WORK_DIR)
print("Workers:", WORKERS, "| batch:", BATCH_SIZE, "| kit scratch:", KIT_SCRATCH_DIR,
      "| archive:", ARCHIVE_BASENAME + "." + ARCHIVE_FORMAT)
print("Kit flags:", " ".join(KIT_RENDER_FLAGS), "| font:", KIT_FIXED_FONT, "| env:", KIT_ENV)


## 2. Install and verify all runtime dependencies

The third-party packages this notebook and its generated worker need are `boto3`, `numpy`, `scipy`, and `Pillow` (worker staging + recolour + resize), plus `matplotlib`, `pandas`, `qrcode`, `pyyaml`, and `wfdb` (the kit's batch-driver imports — `wfdb>=4.3` is REQUIRED: 4.1.x dies at import on pandas 3). The next cell installs every required package up front with `%pip` (unconditional — a fast no-op when already satisfied, so a half-present environment can never slip through and fail later at an `import`), upgrades a stale `wfdb` before it is ever imported, and then verifies each import with an automatic repair install for any straggler.

Before downloading, attach an **EC2 IAM role (instance profile)** to this instance. The role needs `s3:GetObject` on `cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/*` (dataset archive and its `.sha256` sidecar). To upload and verify the output, it also needs `s3:PutObject`, `s3:GetObject`, and `s3:DeleteObject` (preflight probe cleanup, and removing an upload that fails verification) on `cardiosentry/dataset/*`; `s3:AbortMultipartUpload` lets a failed streaming upload clean up its parts. If you get `NoCredentialsError`, use EC2 console → Instances → select this instance → Actions → Security → Modify IAM role, attach the role, then rerun the S3 cell. KMS-encrypted objects additionally require the applicable KMS permissions. The bucket must be reachable from the instance's region/network.


In [ ]:
# Install ALL runtime packages into this kernel's environment BEFORE any
# third-party import. Unconditional (a fast no-op when already satisfied) so a
# half-present environment can never slip through a presence check and fail
# later at `import scipy` with ModuleNotFoundError.
%pip install --quiet boto3 numpy scipy Pillow matplotlib pandas qrcode pyyaml wfdb
import importlib as _package_importlib
import subprocess as _package_subprocess
import sys as _kernel_sys

# boto3 = S3 I/O; numpy/scipy/Pillow = worker staging + recolour + resize;
# matplotlib/pandas/qrcode/pyyaml/wfdb = the kit batch-driver imports.
_module_to_spec = {"boto3": "boto3", "numpy": "numpy", "scipy": "scipy",
                   "PIL": "Pillow", "matplotlib": "matplotlib",
                   "pandas": "pandas", "qrcode": "qrcode",
                   "yaml": "pyyaml", "wfdb": "wfdb>=4.3"}

# wfdb>=4.3 is REQUIRED (4.1.x dies at import on pandas 3): upgrade a stale
# copy BEFORE it is ever imported in this process (upgrading after a failed
# import mixes versions and needs a kernel restart instead).
try:
    from importlib.metadata import version as _dist_version
    _wfdb_have = _dist_version("wfdb")
    _wfdb_old = tuple(int(p) for p in _wfdb_have.split(".")[:2]) < (4, 3)
except Exception:
    _wfdb_have, _wfdb_old = "?", False
if _wfdb_old:
    if "wfdb" in _kernel_sys.modules:
        raise RuntimeError(
            f"wfdb {_wfdb_have} is already imported and too old (need >=4.3): "
            "restart the kernel and re-run from §1.")
    print(f"Upgrading stale wfdb {_wfdb_have} to wfdb>=4.3 ...", flush=True)
    _package_subprocess.check_call(
        [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
         "--retries", "5", "--timeout", "300", "wfdb>=4.3"])
    _package_importlib.invalidate_caches()

# Verify every import; an unimportable module gets one automatic repair
# install (covers detection gaps and broken copies) instead of failing later.
for _mod, _spec in _module_to_spec.items():
    try:
        _package_importlib.import_module(_mod)
    except Exception as exc:
        print(f"{_mod} not importable ({type(exc).__name__}); "
              f"repair-installing {_spec} ...", flush=True)
        _package_subprocess.check_call(
            [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
             "--retries", "5", "--timeout", "300", _spec])
        _package_importlib.invalidate_caches()
        _package_importlib.import_module(_mod)
print("Runtime packages verified:",
      ", ".join(sorted(_module_to_spec)), flush=True)

import csv
from collections import Counter
import hashlib
import importlib
import json
import math
import multiprocessing as mp
import shutil
import sys
import tarfile
import time
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import numpy as np
import scipy
from scipy.io import loadmat as _checked_loadmat
from PIL import Image
from boto3.s3.transfer import TransferConfig

print("Python", sys.version.split()[0], "| boto3", boto3.__version__,
      "| NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| Pillow", Image.__version__)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Download, verify, and extract

The dataset archive and its SHA-256 checksum are read from S3. The downloaded archive is hashed in full before extraction. A matching local archive can be reused. Extraction rejects paths outside the destination and all links and special files. A completion marker allows a clean restart. Keep enough EBS space for the downloaded archive and extracted tree at the same time.


In [ ]:
def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or ":" in parsed.netloc or not parsed.path.strip("/"):
        raise ValueError(f"Expected s3://bucket/key, got {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

from botocore.exceptions import NoCredentialsError, PartialCredentialsError

aws_session = boto3.Session()
try:
    aws_credentials = aws_session.get_credentials()
except (NoCredentialsError, PartialCredentialsError) as exc:
    aws_credentials = None
if aws_credentials is None:
    raise RuntimeError(
        "No AWS credentials are available to this Jupyter kernel. Attach an EC2 IAM "
        "role to this instance: EC2 console > Instances > select instance > Actions > "
        "Security > Modify IAM role. Grant read access to the dataset archive and "
        "checksum, plus write/read access under the S3_UPLOAD_PREFIX. Then rerun this cell."
    )
print("AWS credential source:", aws_credentials.method)
s3 = aws_session.client("s3")
src_bucket, src_key = split_s3_uri(S3_SOURCE)
sha_bucket, sha_key = split_s3_uri(S3_SHA256_SOURCE)
try:
    checksum_object = s3.get_object(Bucket=sha_bucket, Key=sha_key)
except (NoCredentialsError, PartialCredentialsError) as exc:
    raise RuntimeError(
        "AWS credentials could not be used. Check the EC2 instance IAM role and "
        "instance metadata access, then rerun this cell."
    ) from exc
if int(checksum_object["ContentLength"]) > 1024:
    raise ValueError("S3 SHA-256 sidecar is unexpectedly large")
checksum_fields = checksum_object["Body"].read().decode("utf-8").strip().split()
if len(checksum_fields) != 2 or checksum_fields[1].lstrip("*") != LOCAL_ARCHIVE.name:
    raise ValueError("SHA-256 sidecar must name the dataset archive")
expected_sha256 = checksum_fields[0].lower()
if len(expected_sha256) != 64 or any(char not in "0123456789abcdef" for char in expected_sha256):
    raise ValueError("SHA-256 sidecar does not contain a valid digest")

src_info = s3.head_object(Bucket=src_bucket, Key=src_key)
source_bytes = int(src_info["ContentLength"])
source_identity = {"uri": S3_SOURCE, "sha256_uri": S3_SHA256_SOURCE,
                   "sha256": expected_sha256, "bytes": source_bytes,
                   "etag": src_info.get("ETag", "")}
source_sidecar = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".source.json")
archive_matches = (LOCAL_ARCHIVE.is_file() and LOCAL_ARCHIVE.stat().st_size == source_bytes
                   and file_sha256(LOCAL_ARCHIVE) == expected_sha256)
if archive_matches:
    print("Reusing SHA-256 verified source archive:", LOCAL_ARCHIVE)
else:
    partial = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".part")
    partial.unlink(missing_ok=True)
    print(f"Downloading {S3_SOURCE} ({source_bytes / 2**30:.2f} GiB) ...")
    s3.download_file(src_bucket, src_key, str(partial),
                     Config=TransferConfig(**S3_TRANSFER))
    if partial.stat().st_size != source_bytes:
        raise IOError("Downloaded byte count differs from S3 ContentLength")
    if file_sha256(partial) != expected_sha256:
        partial.unlink(missing_ok=True)
        raise IOError("Downloaded archive SHA-256 does not match S3 sidecar")
    partial.replace(LOCAL_ARCHIVE)
    print("Downloaded and SHA-256 verified:", LOCAL_ARCHIVE)
source_sidecar.write_text(json.dumps(source_identity, indent=2) + "\n")

extraction_root = INPUT_DIR.parent.resolve()
marker = extraction_root / ".ecg_clean12_extraction_complete.json"
marker_value = {"source_uri": S3_SOURCE, "archive_sha256": expected_sha256,
                "archive_bytes": source_bytes,
                "archive_mtime": LOCAL_ARCHIVE.stat().st_mtime_ns}
already_extracted = False
if marker.is_file():
    try:
        already_extracted = json.loads(marker.read_text()) == marker_value and INPUT_DIR.is_dir()
    except (OSError, ValueError):
        pass
if already_extracted:
    print("Reusing extracted clean12 ECG tree:", INPUT_DIR)
else:
    marker.unlink(missing_ok=True)
    print("Extracting to", extraction_root, "...")
    member_count = 0
    from pathlib import PurePosixPath
    with tarfile.open(LOCAL_ARCHIVE, "r:gz") as archive:
        for member in archive:
            rel = PurePosixPath(member.name)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts or rel.parts[0] != INPUT_DIR.name:
                raise ValueError(f"Unsafe/unexpected archive path: {member.name!r}")
            dest = extraction_root.joinpath(*rel.parts)
            if not dest.resolve(strict=False).is_relative_to(extraction_root):
                raise ValueError(f"Archive path resolves outside destination: {member.name!r}")
            if member.isdir():
                dest.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                if dest.is_symlink():
                    raise ValueError(f"Refusing to overwrite symlink: {dest}")
                dest.parent.mkdir(parents=True, exist_ok=True)
                stream = archive.extractfile(member)
                if stream is None:
                    raise IOError(f"Cannot read archive member {member.name}")
                with stream, dest.open("wb") as out:
                    shutil.copyfileobj(stream, out, length=1024 * 1024)
                if dest.stat().st_size != member.size:
                    raise IOError(f"Incomplete extracted member {member.name}")
                member_count += 1
            else:
                raise ValueError(f"Links/special files are disallowed: {member.name!r}")
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Archive did not create {INPUT_DIR}")
    marker.write_text(json.dumps(marker_value, indent=2) + "\n")
    print(f"Extracted {member_count:,} files")


## 4. Kit bundle: verify, extract, patch-gate

The `ecg_image_v1_code.tar.gz` bundle (plus its `.sha256` sidecar) is hashed in full before extraction — the same sidecar-pinned discipline as the source archive. Rendering refuses to start unless every `patch_kit` marker is present and the fixed `Arial.ttf` ships in `Fonts/`. A completion marker allows a clean restart. Candidates, in order: the Jupyter working directory (launch Jupyter from the folder holding this notebook and the two kit files), `WORK_DIR/bundle/`, then an available `dataset_generation_code/` copy.
If no local copy exists, the bundle is pulled from S3 (`S3_KIT_SOURCE`, §1) when you have uploaded both kit objects there.


In [ ]:
KIT_BUNDLE = None
_kit_search = [Path.cwd() / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "ecg_image_v1_code.tar.gz"]
for _ancestor in [Path.cwd(), *list(Path.cwd().parents)[:4]]:
    _kit_search.append(_ancestor / "dataset_generation_code" / "ecg_image_v1_code.tar.gz")
for _cand in _kit_search:
    if _cand.is_file():
        KIT_BUNDLE = _cand
        break
if KIT_BUNDLE is None:
    # No local copy — pull the kit bundle from S3 with the same sidecar-pinned
    # discipline as the source archive, then fall through to the checks below.
    try:
        print("No local kit bundle; downloading from", S3_KIT_SOURCE, "...", flush=True)
        _kit_dest = WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz"
        _kit_dest.parent.mkdir(parents=True, exist_ok=True)
        _kb, _kk = split_s3_uri(S3_KIT_SOURCE)
        _sb, _sk = split_s3_uri(S3_KIT_SHA256_SOURCE)
        _sobj = s3.get_object(Bucket=_sb, Key=_sk)
        if int(_sobj["ContentLength"]) > 1024:
            raise ValueError("Kit SHA-256 sidecar is unexpectedly large")
        _raw = _sobj["Body"].read()
        (_kit_dest.parent / (_kit_dest.name + ".sha256")).write_bytes(_raw)
        _fields = _raw.decode("utf-8").strip().split()
        if len(_fields) != 2 or _fields[1].lstrip("*") != _kit_dest.name:
            raise ValueError("Kit SHA-256 sidecar must name the kit bundle")
        _want = _fields[0].lower()
        if len(_want) != 64 or any(c not in "0123456789abcdef" for c in _want):
            raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
        _kinfo = s3.head_object(Bucket=_kb, Key=_kk)
        _kbytes = int(_kinfo["ContentLength"])
        if not (_kit_dest.is_file() and _kit_dest.stat().st_size == _kbytes
                and file_sha256(_kit_dest) == _want):
            _kpart = _kit_dest.with_name(_kit_dest.name + ".part")
            _kpart.unlink(missing_ok=True)
            print(f"Downloading {S3_KIT_SOURCE} ({_kbytes / 2**20:.1f} MiB) ...", flush=True)
            s3.download_file(_kb, _kk, str(_kpart),
                             Config=TransferConfig(**S3_TRANSFER))
            if _kpart.stat().st_size != _kbytes:
                raise IOError("Downloaded kit byte count differs from S3 ContentLength")
            if file_sha256(_kpart) != _want:
                _kpart.unlink(missing_ok=True)
                raise IOError("Downloaded kit SHA-256 does not match S3 sidecar")
            _kpart.replace(_kit_dest)
            print("Downloaded and SHA-256 verified:", _kit_dest, flush=True)
        else:
            print("Reusing SHA-256 verified kit bundle:", _kit_dest, flush=True)
        KIT_BUNDLE = _kit_dest
    except Exception as exc:
        raise FileNotFoundError(
            "ecg_image_v1_code.tar.gz not found locally and the S3 fallback failed "
            f"({type(exc).__name__}: {exc}). Searched: " +
            ", ".join(str(p) for p in _kit_search) +
            f"; S3: {S3_KIT_SOURCE}") from exc
_kit_sidecar = Path(str(KIT_BUNDLE) + ".sha256")
if not _kit_sidecar.is_file():
    raise FileNotFoundError(f"Kit SHA-256 sidecar missing: {_kit_sidecar}")
_kit_want = _kit_sidecar.read_text().split()[0].lower()
if len(_kit_want) != 64 or any(c not in "0123456789abcdef" for c in _kit_want):
    raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
KIT_BUNDLE_SHA256 = file_sha256(KIT_BUNDLE)
if KIT_BUNDLE_SHA256 != _kit_want:
    raise IOError("Kit bundle SHA-256 does not match its sidecar — re-transfer the file")
print("Kit bundle verified:", KIT_BUNDLE)

KIT_ROOT = WORK_DIR / "kit"
_kit_marker = KIT_ROOT / ".kit_extraction_complete.json"
_kit_marker_value = {"bundle_sha256": KIT_BUNDLE_SHA256,
                     "bundle_bytes": KIT_BUNDLE.stat().st_size}
_kit_ready = False
if _kit_marker.is_file():
    try:
        _kit_ready = json.loads(_kit_marker.read_text()) == _kit_marker_value and KIT_ROOT.is_dir()
    except (OSError, ValueError):
        pass
if _kit_ready:
    print("Reusing extracted kit:", KIT_ROOT)
else:
    _kit_marker.unlink(missing_ok=True)
    print("Extracting kit to", KIT_ROOT, "...")
    shutil.rmtree(KIT_ROOT, ignore_errors=True)
    with tarfile.open(KIT_BUNDLE, "r:gz") as _archive:
        try:
            _archive.extractall(KIT_ROOT, filter="data")
        except TypeError:  # Python without PEP 706 filters; bundle is SHA-256 verified
            _archive.extractall(KIT_ROOT)
    _kit_marker.write_text(json.dumps(_kit_marker_value, indent=2) + "\n")
    print("Kit extracted")

KIT_DIR = str(KIT_ROOT / "ecg_image_v1_code" / "ecg-image-generator")
_KIT_SCRIPT = Path(KIT_DIR) / "gen_ecg_images_from_data_batch.py"
if not _KIT_SCRIPT.is_file():
    raise FileNotFoundError(f"Kit batch driver missing: {_KIT_SCRIPT}")
if not (Path(KIT_DIR) / "Fonts" / KIT_FIXED_FONT).is_file():
    raise FileNotFoundError(f"Fixed font {KIT_FIXED_FONT} missing from kit Fonts/")

# Patch gate (PLAN §6): refuse to render from unpatched kit code.
_kit_need = {
    "ecg_plot.py": ["PATCHED by image_pipeline/patch_kit.py",
                    "PATCHED-LEADNAME", "PATCHED-RHYTHMNAME",
                    "PATCHED-INKWIDTH", "PATCHED-LAYOUT", "PATCHED-PAPER"],
    "extract_leads.py": ["PATCHED-COLUMNS"],
    "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
    "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
    "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"],
}
_kit_missing = []
for _name, _markers in _kit_need.items():
    _text = (Path(KIT_DIR) / _name).read_text()
    for _mk in _markers:
        if _mk not in _text:
            _kit_missing.append(f"{_name}:{_mk}")
if _kit_missing:
    raise RuntimeError(f"Kit at {KIT_DIR} is not patched "
                       f"({len(_kit_missing)} markers missing, e.g. {_kit_missing[:3]}); refusing to render")
print(f"Kit patch gate OK ({KIT_DIR})")


## 5. Discover and validate clean12 records

Records come from the archive's `manifest.csv` (`record_path` relative to `ECG_500Hz_clean12/`), never from a glob. Every listed header must exist, be unique, and be 12 leads x 5,000 samples at 500 Hz, and the per-condition label totals must equal `_checks/final_counts.csv` from the download's step 4 verify. Output PNG paths mirror the record paths under `images/`. The source manifest is copied into the image package as `record_manifest.csv` (labels, `patient_key`, `other_findings`, signal sha256), with the selection report, final counts, and documentation, so every image can be traced to its ECG, labels, and patient.


In [ ]:
with (INPUT_DIR / "manifest.csv").open(newline="", encoding="utf-8") as handle:
    source_rows = list(csv.DictReader(handle))
records = [row["record_path"] for row in source_rows]
if len(records) != EXPECTED_RECORDS:
    raise ValueError(f"manifest.csv lists {len(records):,} records; expected {EXPECTED_RECORDS:,}")
if len(set(records)) != len(records):
    raise RuntimeError("Duplicate record_path in manifest.csv")

from pathlib import PurePosixPath
site_counts = Counter()
patients = set()
label_totals = Counter()
for row in source_rows:
    rel = PurePosixPath(row["record_path"])
    if (rel.is_absolute() or ".." in rel.parts or rel.suffix != ".hea"
            or rel.parts[0] != row["site"] or rel.parts[1] != "WFDB"):
        raise ValueError(f"Unexpected record_path: {row['record_path']!r}")
    header = INPUT_DIR.joinpath(*rel.parts)
    if not header.is_file():
        raise FileNotFoundError(f"Manifest record missing from extracted tree: {rel}")
    first = header.read_text(encoding="utf-8", errors="replace").split("\n", 1)[0].split()
    if (len(first) < 4 or first[1] != "12"
            or float(first[2].split("/")[0]) != EXPECTED_FS or int(first[3]) != EXPECTED_SAMPLES):
        raise ValueError(f"{rel}: header {' '.join(first[:4])!r} is not 12 x {EXPECTED_SAMPLES} @ {EXPECTED_FS} Hz")
    site_counts[row["site"]] += 1
    patients.add(row["patient_key"])
    for condition in CONDITIONS:
        label_totals[condition] += int(row[condition])

with (INPUT_DIR / "_checks" / "final_counts.csv").open(newline="", encoding="utf-8") as handle:
    verified_totals = {row["condition"]: int(row["total"]) for row in csv.DictReader(handle)}
if verified_totals != dict(label_totals):
    raise ValueError(f"Label totals {dict(label_totals)} differ from _checks/final_counts.csv {verified_totals}")
site_counts = dict(sorted(site_counts.items()))
print(f"Discovered {len(records):,} records from {len(patients):,} patients")
print("Records per site:", site_counts)
print("Records per condition:", dict(label_totals))

for source_name, package_name in (("manifest.csv", "record_manifest.csv"),
                                  ("selection_report.json", "selection_report.json"),
                                  ("_checks/final_counts.csv", "final_counts.csv"),
                                  ("docs/README.md", "README.md"),
                                  ("docs/SUMMARY.md", "SUMMARY.md")):
    shutil.copy2(INPUT_DIR / source_name, OUTPUT_DIR / package_name)

render_config = {
    "render_version": RENDER_VERSION,
    "renderer": "ecg-image-kit",
    "dataset": "HEEDB clean12",
    "source_uri": S3_SOURCE,
    "source_sha256": expected_sha256,
    "source_etag": source_identity["etag"],
    "source_bytes": source_bytes,
    "record_count": len(records),
    "site_record_counts": site_counts,
    "condition_record_counts": dict(label_totals),
    "kit_bundle_sha256": KIT_BUNDLE_SHA256,
    "kit_seed": KIT_SEED,
    "kit_fixed_font": KIT_FIXED_FONT,
    "kit_render_flags": KIT_RENDER_FLAGS,
    "kit_env": KIT_ENV,
    "kit_dir": KIT_DIR,
    "kit_python": sys.executable,
    "kit_scratch_dir": str(KIT_SCRATCH_DIR),
    "image_format": IMAGE_FORMAT,
    "final_image_width": FINAL_IMAGE_WIDTH,
    "final_image_height": FINAL_IMAGE_HEIGHT,
    "png_compress_level": PNG_COMPRESS_LEVEL,
    "max_duration_s": MAX_DURATION_S,
    "layout": "3x4_rhythm",
    "n_columns": 4,
    "column_duration_s": MAX_DURATION_S / 4,
    "rhythm_lead": "II",
    "lead_layout": [["I", "aVR", "V1", "V4"], ["II", "aVL", "V2", "V5"], ["III", "aVF", "V3", "V6"]],
    "lead_order": ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"],
}
config_bytes = json.dumps(render_config, sort_keys=True).encode()
config_hash = hashlib.sha256(config_bytes).hexdigest()
lock_file = OUTPUT_DIR / ".render_config_sha256"
if lock_file.is_file() and lock_file.read_text().strip() != config_hash:
    raise RuntimeError("Existing images use different render settings. Use a fresh OUTPUT_DIR or restore the prior settings.")
lock_file.write_text(config_hash + "\n")
(OUTPUT_DIR / "config.json").write_text(json.dumps(render_config, indent=2) + "\n")
(OUTPUT_DIR / "images").mkdir(exist_ok=True)


## 6. Self-contained process worker

The worker is written from this cell into the working directory. Spawned Jupyter processes import that module once. Each batch is staged to canonical scratch WFDB `.dat` files (fmt 16, gain 1000, int16 round-trip verified — dodging the kit's `.mat` gain bug) and rendered by the real ECG-image-kit batch driver with the reference flags/env/fixed font; grey masters are recoloured to the canonical clean red grid and resized to 1024x768. Each PNG is written through a temporary file and atomic rename. Workers write disjoint image paths.


In [ ]:
WORKER_SOURCE = r'''
"""Clean ECG worker: render with the real ECG-image-kit (PLAN §6).

Each batch is staged to canonical scratch WFDB .dat files (fmt 16, gain
1000, canonical lead order, int16 round-trip verified — this dodges the
kit's .mat gain bug, PLAN §0.5) and rendered by the kit batch driver
with the reference flags/env/fixed font. Grey masters are recoloured to
the canonical clean red grid (fixed LUT, PLAN §10) and resized LANCZOS
to 994x768 + 15 px paper-colour pads -> final 1024x768 PNG (PLAN §7.3).

Self-contained: stdlib + numpy + scipy + Pillow only. The kit itself
runs in a subprocess (kit_python, which carries the kit's own deps).
"""
import math
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from PIL import Image

LEADS = ("I", "II", "III", "aVR", "aVL", "aVF",
         "V1", "V2", "V3", "V4", "V5", "V6")
KIT_SCRIPT = "gen_ecg_images_from_data_batch.py"
KIT_TIMEOUT_S = 3600

# Clean-view recolour: fixed "red" palette (PLAN §10.3; port of
# v1_support.variants._lut with ink=(20,20,20), paper=red paper, sat=1.0).
GRID_MAJOR = (178.0, 86.0, 84.0)
GRID_MINOR = (228.0, 180.0, 176.0)
GRID_PAPER = (253.0, 249.0, 247.0)
GRID_INK = (20.0, 20.0, 20.0)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250

FINAL_W, FINAL_H = 1024, 768
RESIZE_H = 768
MASTER_W, MASTER_H = 2200, 1700

CFG = None
INPUT = None
OUTPUT = None
_LUT = None


class InvalidRecordError(ValueError):
    pass


def init_worker(config, input_dir, output_dir):
    global CFG, INPUT, OUTPUT
    CFG = config
    INPUT = Path(input_dir)
    OUTPUT = Path(output_dir)
    for key in ("kit_dir", "kit_python", "kit_seed", "kit_fixed_font",
                "kit_render_flags", "kit_env", "kit_scratch_dir",
                "max_duration_s", "png_compress_level"):
        if key not in CFG:
            raise ValueError(f"render_config missing {key!r}")
    if not (Path(CFG["kit_dir"]) / KIT_SCRIPT).is_file():
        raise ValueError("kit batch driver missing — rerun the kit-bundle cell")


def _lut():
    major = np.asarray(GRID_MAJOR, dtype=np.float32)
    minor = np.asarray(GRID_MINOR, dtype=np.float32)
    paper = np.asarray(GRID_PAPER, dtype=np.float32)
    ink = np.asarray(GRID_INK, dtype=np.float32)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * 1.0, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL,
                   GRID_MINOR_LEVEL, GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch],
                       paper[ch], min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255)


def recolour_clean(master):
    """Fixed clean-red recolour of a grey kit master.

    The float LUT is truncated to uint8 once (what the kit-v1 per-pixel
    `_LUT[grey].astype(np.uint8)` did) and applied through Pillow point
    tables: bit-identical output, ~7x cheaper per image.
    """
    global _LUT
    if _LUT is None:
        lut = _lut().astype(np.uint8)
        _LUT = [lut[:, ch].tolist() for ch in range(3)]
    grey = master.convert("RGB").convert("L")
    return Image.merge("RGB", [grey.point(table) for table in _LUT])


def _paper_colour(arr):
    # 90th percentile of corner patches (not median): antialiased grid-halo
    # pixels drag a median toward tint; p90 recovers the paper white.
    h, w = arr.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        arr[:k, :k].reshape(-1, 3), arr[:k, -k:].reshape(-1, 3),
        arr[-k:, :k].reshape(-1, 3), arr[-k:, -k:].reshape(-1, 3)])
    return tuple(int(v) for v in np.percentile(patches, 90, axis=0))


def to_final(img):
    """LANCZOS fit to height 768 + symmetric paper-colour pad to 1024x768."""
    w, h = img.size
    scale = RESIZE_H / h
    new_w = round(w * scale)
    small = img.resize((new_w, RESIZE_H), Image.LANCZOS)
    if abs((w / h) - (new_w / RESIZE_H)) > 0.5 / RESIZE_H + 1e-9:
        raise InvalidRecordError(
            f"aspect drift: source {w}x{h} -> {new_w}x{RESIZE_H}")
    pad_total = FINAL_W - new_w
    if pad_total < 0:
        raise InvalidRecordError(
            f"resized width {new_w} exceeds final {FINAL_W}")
    left = pad_total // 2
    fill = _paper_colour(np.asarray(small.convert("RGB"), dtype=np.float32))
    canvas = Image.new("RGB", (FINAL_W, FINAL_H), fill)
    canvas.paste(small.convert("RGB"), (left, 0))
    if canvas.size != (FINAL_W, FINAL_H):
        raise InvalidRecordError(f"final size {canvas.size}")
    return canvas


def parse_header(path):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError as exc:
        raise InvalidRecordError(f"Unreadable header: {exc}") from exc
    if not lines:
        raise InvalidRecordError("Empty header")
    first = lines[0].split()
    if len(first) < 4 or "/" in first[0] or first[0] != path.stem:
        raise InvalidRecordError("Malformed/multi-segment record line")
    try:
        nleads = int(first[1])
        fs = float(first[2].split("/")[0])
        nsamp = int(first[3])
    except ValueError as exc:
        raise InvalidRecordError("Invalid record dimensions") from exc
    if nleads != 12 or not math.isfinite(fs) or fs <= 0 or nsamp <= 0:
        raise InvalidRecordError(
            f"Unsupported dimensions: {nleads} leads, fs={fs}, n={nsamp}")
    duration = nsamp / fs
    if duration > CFG["max_duration_s"] + 1e-6:
        raise InvalidRecordError(
            f"Duration {duration:.4f}s exceeds configured maximum")
    rows = []
    comments = {}
    for line in lines[1:]:
        if line.startswith("#"):
            body = line[1:].strip()
            if ":" in body:
                k, v = body.split(":", 1)
                comments[k.strip()] = v.strip()
            continue
        if not line.strip():
            continue
        f = line.split()
        if len(f) < 9:
            raise InvalidRecordError("Malformed signal line")
        name = f[0]
        if name != Path(name).name or name in (".", "..", "~"):
            raise InvalidRecordError("Unsafe signal filename")
        fmt = f[1]
        if fmt not in ("16", "16+24", "16x1+24"):
            raise InvalidRecordError(f"Unsupported signal format {fmt}")
        gain_field = f[2]
        if not gain_field.endswith("/mV"):
            raise InvalidRecordError(f"Expected /mV units, got {gain_field}")
        try:
            gain = float(gain_field.split("(")[0].split("/")[0])
            adc_zero = int(f[4])
        except ValueError as exc:
            raise InvalidRecordError("Invalid gain or ADC zero") from exc
        if not math.isfinite(gain) or gain <= 0:
            raise InvalidRecordError("Nonpositive/nonfinite ADC gain")
        if "(" in gain_field and gain_field.split("(", 1)[1].split(")", 1)[0] != str(adc_zero):
            raise InvalidRecordError("Header baseline/ADC zero disagree")
        rows.append({"file": name, "fmt": fmt, "gain": gain,
                     "zero": adc_zero, "lead": f[-1]})
    if len(rows) != 12:
        raise InvalidRecordError("Need exactly 12 signal lines")
    upper = {lead.upper(): lead for lead in LEADS}
    names = []
    for r in rows:
        key = r["lead"].strip().upper()
        if key not in upper:
            raise InvalidRecordError(f"Non-canonical lead {r['lead']!r}")
        names.append(upper[key])
    if sorted(names) != sorted(LEADS):
        raise InvalidRecordError("Lead set is not the canonical 12")
    if len({r["file"] for r in rows}) != 1 or len({r["fmt"] for r in rows}) != 1:
        raise InvalidRecordError("Multiple signal files/formats not supported")
    return rows, names, fs, nsamp, duration, comments


def load_mv(path, rows, names, nsamp):
    fmt = rows[0]["fmt"]
    sig_path = path.parent / rows[0]["file"]
    if not sig_path.is_file():
        raise InvalidRecordError(f"Missing signal file {rows[0]['file']}")
    if fmt == "16":
        n = 12 * nsamp
        raw = np.fromfile(str(sig_path), dtype="<i2", count=n).astype(np.int64)
        if raw.size != n:
            raise InvalidRecordError(
                f".dat holds {raw.size} samples, need {n}")
        raw = raw.reshape(nsamp, 12).T
    else:
        try:
            arr = np.asarray(loadmat(str(sig_path), variable_names=["val"])["val"])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            raise InvalidRecordError(f"Unreadable MAT val: {exc}") from exc
        if arr.shape == (nsamp, 12):
            arr = arr.T
        if arr.shape != (12, nsamp) or arr.dtype.kind not in "iu":
            raise InvalidRecordError(
                f"Invalid MAT val shape/type {arr.shape}/{arr.dtype}")
        raw = arr.astype(np.int64)
    if raw.shape != (12, nsamp):
        raise InvalidRecordError(f"Unexpected signal shape {raw.shape}")
    pos = {name: i for i, name in enumerate(names)}
    order = [pos[lead] for lead in LEADS]
    gains = np.array([rows[pos[lead]]["gain"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    zeros = np.array([rows[pos[lead]]["zero"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    # Necessary unit conversion only: digital ADC counts -> physical millivolts.
    mv = (raw[order, :].astype(np.float64) - zeros) / gains
    if not np.isfinite(mv).all():
        raise InvalidRecordError("Nonfinite physical signal")
    return mv


def stage_record(record_id, mv, fs, comments, staged_dir):
    """Write canonical scratch .dat + .hea for the kit; verify round-trip."""
    if mv.shape[0] != 12:
        raise InvalidRecordError(f"shape {mv.shape}, need (12, N)")
    n = mv.shape[1]
    raw = np.round(mv * 1000.0)
    if raw.min() < -32768 or raw.max() > 32767:
        raise InvalidRecordError("signal exceeds int16 at gain 1000")
    staged_dir.mkdir(parents=True, exist_ok=True)
    hea = staged_dir / f"{record_id}.hea"
    dat = staged_dir / f"{record_id}.dat"
    np.ascontiguousarray(raw.astype("<i2").T).tofile(str(dat))
    rec_fs = int(fs) if float(fs).is_integer() else fs
    lines = [f"{record_id} 12 {rec_fs} {n}"]
    for lead in LEADS:
        lines.append(f"{record_id}.dat 16 1000.0(0)/mV 16 0 0 0 0 {lead}")
    for key in ("Age", "Sex"):
        if comments.get(key):
            lines.append(f"# {key}: {comments[key]}")
    hea.write_text("\n".join(lines) + "\n")
    back = np.fromfile(str(dat), dtype="<i2",
                       count=12 * n).astype(np.int64).reshape(n, 12).T
    if back.shape != raw.shape or not np.array_equal(back, raw.astype(np.int64)):
        raise InvalidRecordError("staging round-trip mismatch")
    return hea


def run_kit(staged_dir, render_dir):
    """Invoke the kit batch driver once for a staged directory."""
    kit_dir = Path(CFG["kit_dir"])
    if not (kit_dir / KIT_SCRIPT).is_file():
        raise RuntimeError(f"kit script missing: {kit_dir / KIT_SCRIPT}")
    render_dir.mkdir(parents=True, exist_ok=True)
    # NOTE: absolute paths are mandatory — the kit subprocess runs with
    # cwd=kit_dir, so relative -i/-o would resolve inside the kit tree.
    cmd = [CFG["kit_python"], KIT_SCRIPT,
           "-i", str(Path(staged_dir).resolve()),
           "-o", str(Path(render_dir).resolve()),
           "-se", str(CFG["kit_seed"]), *CFG["kit_render_flags"]]
    env = dict(os.environ)
    env.update(CFG.get("kit_env", {}))
    env["ECG_FIXED_FONT"] = CFG["kit_fixed_font"]
    # one numerical thread per renderer; never the GPU
    env.update({"CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1",
                "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
    proc = subprocess.run(cmd, cwd=str(kit_dir), capture_output=True,
                          text=True, timeout=KIT_TIMEOUT_S, env=env)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-3:]
        raise RuntimeError(f"kit batch failed "
                           f"({staged_dir} -> {render_dir}): {'; '.join(tail)}")


def collect_pngs(render_dir, record_ids):
    """Map record_id -> rendered PNG. The kit names outputs <record>-0.png."""
    found = {}
    for png in Path(render_dir).rglob("*.png"):
        stem = png.stem  # <record_id>-0
        rid = stem[:-2] if stem.endswith("-0") else stem
        if rid in record_ids:
            found[rid] = png
    missing = sorted(set(record_ids) - set(found))
    if missing:
        raise RuntimeError(f"kit produced no PNG for {len(missing)} records, "
                           f"e.g. {missing[:5]}")
    return found


def clean_render_dir(render_dir):
    """Remove kit side products; only *.png may remain."""
    for pat in ("*.hea", "*.dat", "*.json", "*.csv"):
        for p in Path(render_dir).rglob(pat):
            try:
                p.unlink()
            except OSError:
                pass


def record_id_for(rel):
    stem = Path(rel).with_suffix("").as_posix()
    rid = stem.replace("/", "__").replace("\\", "__")
    if not rid or rid in (".", "..") or "/" in rid or len(rid) > 180:
        raise InvalidRecordError(f"Unusable record id for {rel!r}")
    return rid


def _base_row(rel):
    return {"source_header": rel,
            "image_path": str(Path("images") / Path(rel).with_suffix(".png")),
            "status": "", "error": "", "sample_rate_hz": "", "samples": "",
            "duration_s": "", "image_width": "", "image_height": "",
            "image_bytes": ""}


def process_batch(batch):
    scratch = (Path(CFG["kit_scratch_dir"])
               / f"w{os.getpid()}-{int(time.time() * 1000) % 100000000}")
    staged = scratch / "staged"
    render = scratch / "render"
    rows = {}
    todo = {}
    for rel in batch:
        base = _base_row(rel)
        rows[rel] = base
        header = INPUT / rel
        try:
            prows, names, fs, nsamp, duration, comments = parse_header(header)
            base.update(sample_rate_hz=fs, samples=nsamp,
                        duration_s=round(duration, 6))
            out = OUTPUT / base["image_path"]
            if out.is_file() and out.stat().st_size > 100:
                try:
                    with Image.open(out) as existing:
                        ok = existing.size == (FINAL_W, FINAL_H)
                        existing.verify()
                    if ok:
                        base.update(status="skipped", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                        continue
                except (OSError, ValueError):
                    pass
                # A mismatched or damaged existing image is regenerated.
            mv = load_mv(header, prows, names, nsamp)
            rid = record_id_for(rel)
            if rid in todo:
                raise InvalidRecordError(f"Duplicate staged id {rid}")
            todo[rid] = (rel, mv, fs, comments)
        except InvalidRecordError as exc:
            base.update(status="invalid", error=str(exc)[:1000])
        except Exception as exc:
            base.update(status="failed",
                        error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            for rid, (rel, mv, fs, comments) in todo.items():
                if rows[rel]["status"]:
                    continue
                try:
                    stage_record(rid, mv, fs, comments, staged)
                except InvalidRecordError as exc:
                    rows[rel].update(status="invalid", error=str(exc)[:1000])
            staged_ids = [rid for rid in todo if not rows[todo[rid][0]]["status"]]
            if staged_ids:
                try:
                    run_kit(staged, render)
                    got = collect_pngs(render, set(staged_ids))
                except Exception as exc:
                    for rid in staged_ids:
                        rows[todo[rid][0]].update(
                            status="failed",
                            error=f"{type(exc).__name__}: {exc}"[:1000])
                    got = {}
                for rid, png in got.items():
                    rel = todo[rid][0]
                    base = rows[rel]
                    try:
                        with Image.open(png) as master:
                            if master.size != (MASTER_W, MASTER_H):
                                raise InvalidRecordError(
                                    f"master size {master.size}, "
                                    f"need {(MASTER_W, MASTER_H)}")
                            view = recolour_clean(master)
                        final = to_final(view)
                        out = OUTPUT / base["image_path"]
                        out.parent.mkdir(parents=True, exist_ok=True)
                        temp = out.with_name(out.name + f".{os.getpid()}.part")
                        try:
                            final.save(temp, format="PNG",
                                       compress_level=CFG["png_compress_level"])
                            temp.replace(out)
                        finally:
                            try:
                                temp.unlink(missing_ok=True)
                            except OSError:
                                pass
                            final.close()
                        base.update(status="success", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                    except InvalidRecordError as exc:
                        base.update(status="invalid", error=str(exc)[:1000])
                    except Exception as exc:
                        base.update(status="failed",
                                    error=f"{type(exc).__name__}: {exc}"[:1000])
                clean_render_dir(render)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[rel] for rel in batch]


def process_one(rel):
    return process_batch([rel])[0]


def ping():
    """Zero-cost import probe: lets §7 verify a spawn-context pool child
    can import this module without rendering anything."""
    return {"ok": True, "leads": list(LEADS),
            "final": (FINAL_W, FINAL_H), "master": (MASTER_W, MASTER_H)}

'''

worker_path = WORK_DIR / "ecg_clean_worker.py"
compile(WORKER_SOURCE, str(worker_path), "exec")
worker_path.write_text(WORKER_SOURCE)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
# Spawn-context pool children (§8) start a fresh interpreter: they inherit
# os.environ but NOT this sys.path edit, and both the pool initializer and
# every task import ecg_clean_worker by module reference. PYTHONPATH carries
# WORK_DIR across, otherwise every child dies with ModuleNotFoundError.
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = str(WORK_DIR) + (
        (os.pathsep + os.environ["PYTHONPATH"]) if os.environ.get("PYTHONPATH") else "")
importlib.invalidate_caches()
import ecg_clean_worker
ecg_clean_worker = importlib.reload(ecg_clean_worker)
print("Worker module ready:", worker_path)


## 7. Preflight / smoke test (run before the overnight job)

Run this cell after the worker cell and before the full generate. It takes ~1-2 min and checks everything that would otherwise wake you up at 3am: imports, kit bundle + patch gate + fixed font, AWS credentials and source reachability, **upload write permission** (tiny probe object, deleted afterwards), disk space, worker import, CPU/SMT and RAM-scratch checks, and one real end-to-end record through the kit (`stage -> run_kit -> recolour -> 1024x768 PNG`). The test image lands in the real `OUTPUT_DIR` so the full run reuses it as `skipped`.

Do not start §8 until this cell prints `PREFLIGHT PASS`. If it prints `PREFLIGHT FAIL`, fix the listed item(s) and re-run this cell only.



In [ ]:
# §7 preflight: fail fast before the overnight run (~1-2 min).
import time as _pf_time

_pf_failures = []
_pf_warnings = []

def _pf_check(name, fn):
    try:
        detail = fn()
        print(f"  [PASS] {name}" + (f" — {detail}" if detail else ""), flush=True)
        return detail
    except Exception as exc:
        print(f"  [FAIL] {name}: {type(exc).__name__}: {exc}", flush=True)
        _pf_failures.append(f"{name}: {type(exc).__name__}: {exc}")
        return None

print("Preflight — fast checks first ...", flush=True)

def _pf_imports():
    import boto3 as _b, numpy as _np, scipy as _sp, PIL as _pil
    from PIL import Image as _I
    import matplotlib as _mpl, pandas as _pd, qrcode as _qr, yaml as _y, wfdb as _w
    return (f"py {sys.version.split()[0]}, boto3 {_b.__version__}, "
            f"np {_np.__version__}, scipy {_sp.__version__}, "
            f"Pillow {_I.__version__}, mpl {_mpl.__version__}, "
            f"pd {_pd.__version__}, wfdb {_w.__version__}")
_pf_check("imports (boto3/numpy/scipy/Pillow/matplotlib/pandas/qrcode/yaml/wfdb)", _pf_imports)

def _pf_paths():
    assert INPUT_DIR.is_dir(), f"INPUT_DIR missing: {INPUT_DIR}"
    assert OUTPUT_DIR.is_dir(), f"OUTPUT_DIR missing: {OUTPUT_DIR}"
    drv = Path(render_config["kit_dir"]) / "gen_ecg_images_from_data_batch.py"
    assert drv.is_file(), f"kit driver missing: {drv}"
    font = Path(render_config["kit_dir"]) / "Fonts" / render_config["kit_fixed_font"]
    assert font.is_file(), f"fixed font missing: {font}"
    n = len(records)
    assert n > 0, "no records discovered — rerun §5"
    return f"{n:,} records, kit {render_config['kit_dir']}, font {render_config['kit_fixed_font']}"
_pf_check("paths (INPUT/OUTPUT/kit driver/font/records)", _pf_paths)

def _pf_patch_gate():
    need = {"ecg_plot.py": ["PATCHED-LEADNAME", "PATCHED-RHYTHMNAME", "PATCHED-INKWIDTH",
                            "PATCHED-LAYOUT", "PATCHED-PAPER"],
            "extract_leads.py": ["PATCHED-COLUMNS"],
            "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
            "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
            "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"]}
    missing = [f"{n}:{m}" for n, ms in need.items()
               for m in ms if m not in (Path(render_config["kit_dir"]) / n).read_text()]
    assert not missing, f"unpatched kit: {missing[:4]}"
    return "all 11 markers present"
_pf_check("kit patch gate", _pf_patch_gate)

def _pf_workers():
    import os as _os
    cpu = _os.cpu_count() or 0
    assert WORKERS >= 1 and BATCH_SIZE >= 1, "WORKERS/BATCH_SIZE misconfigured"
    return f"WORKERS={WORKERS} (cpu={cpu}), BATCH_SIZE={BATCH_SIZE}"
_pf_check("workers/cpu", _pf_workers)

def _pf_cpu():
    model = "?"
    with open("/proc/cpuinfo") as _fh:
        for _line in _fh:
            if _line.startswith("model name"):
                model = _line.split(":", 1)[1].strip()
                break
    smt = Path("/sys/devices/system/cpu/smt/active")
    smt_on = smt.is_file() and smt.read_text().strip() == "1"
    if smt_on:
        _pf_warnings.append("SMT is on: vCPUs are hyperthreads, so WORKERS=vCPUs gives ~1.1-1.3x "
                            "a core count, not 2x. Prefer c8a/c7a (1 vCPU = 1 core).")
    return f"{model}, SMT {'on' if smt_on else 'off'}"
_pf_check("cpu model / SMT", _pf_cpu)

def _pf_scratch():
    KIT_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage(KIT_SCRATCH_DIR).free / 2**30
    need_gib = WORKERS * 0.1  # ~50 MB per in-flight batch of 48
    assert free_gib >= need_gib, f"{free_gib:.1f} GiB free at {KIT_SCRATCH_DIR}, need {need_gib:.1f}"
    return f"{KIT_SCRATCH_DIR} ({free_gib:.1f} GiB free)"
_pf_check("kit scratch space", _pf_scratch)

def _pf_disk():
    du = shutil.disk_usage(WORK_DIR)
    free_gib = du.free / 2**30
    # At this point source (5.5) + extracted tree (13.5) are already on disk;
    # still needed: images (~104) + metadata. §9 streams the TAR to S3, so
    # no local archive space is needed; kit scratch is in /dev/shm.
    need_gib = 110.0
    msg = f"{free_gib:.0f} GiB free, need ~{need_gib:.0f} GiB from here"
    assert free_gib >= need_gib, msg + " — enlarge the EBS volume before §8"
    return msg
_pf_check("disk space on WORK_DIR volume", _pf_disk)

def _pf_aws_source():
    creds = aws_session.get_credentials()
    assert creds is not None, "no AWS credentials — attach the EC2 IAM role and rerun §3"
    info = s3.head_object(Bucket=src_bucket, Key=src_key)
    assert int(info["ContentLength"]) == source_bytes, "S3 source size changed since §3"
    return f"creds via {creds.method}, source {source_bytes / 2**30:.2f} GiB reachable"
_pf_check("AWS creds + source reachable", _pf_aws_source)

def _pf_upload_probe():
    up = urlparse(S3_UPLOAD_PREFIX)
    assert up.scheme == "s3" and up.netloc, f"bad S3_UPLOAD_PREFIX: {S3_UPLOAD_PREFIX!r}"
    key = "/".join(p for p in (up.path.strip("/"), ".preflight_write_probe.txt") if p)
    body = f"preflight {datetime.now(timezone.utc).isoformat()}\n".encode()
    try:
        s3.put_object(Bucket=up.netloc, Key=key, Body=body)
        head = s3.head_object(Bucket=up.netloc, Key=key)
        assert int(head["ContentLength"]) == len(body), "probe size mismatch"
        return f"write+read OK at s3://{up.netloc}/{key} (deleted after)"
    finally:
        try:
            s3.delete_object(Bucket=up.netloc, Key=key)
        except Exception:
            pass
_pf_check("upload write permission (probe object)", _pf_upload_probe)

def _pf_worker_import():
    assert hasattr(ecg_clean_worker, "process_one"), "worker has no process_one — rerun §6"
    assert hasattr(ecg_clean_worker, "init_worker"), "worker has no init_worker — rerun §6"
    return str(worker_path)
_pf_check("worker module import", _pf_worker_import)

def _pf_pythonpath():
    pp = os.environ.get("PYTHONPATH", "")
    assert str(WORK_DIR) in pp.split(os.pathsep), \
        f"WORK_DIR not on PYTHONPATH ({pp!r}) — rerun §6"
    return "WORK_DIR visible to spawn children"
_pf_check("PYTHONPATH carries WORK_DIR to pool children", _pf_pythonpath)

print("Preflight — spawn-pool child import (same machinery as §8) ...", flush=True)
try:
    _pf_ctx = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=1, mp_context=_pf_ctx) as _pf_ex:
        _pf_pong = _pf_ex.submit(ecg_clean_worker.ping).result(timeout=300)
    assert _pf_pong.get("ok") and tuple(_pf_pong.get("final", ())) == (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT), \
        f"bad pong: {_pf_pong}"
    print(f"  [PASS] spawn-pool ping — child imported worker, pong {_pf_pong}", flush=True)
except Exception as exc:
    print(f"  [FAIL] spawn-pool ping: {type(exc).__name__}: {exc}", flush=True)
    _pf_failures.append(f"spawn-pool ping: {type(exc).__name__}: {exc}")

print("Preflight — one real record end-to-end ...", flush=True)
_pf_single_secs = None
_pf_single_rel = None
try:
    ecg_clean_worker.init_worker(render_config, str(INPUT_DIR), str(OUTPUT_DIR))
    _pf_row = None
    for _rel in list(records[:3]):
        t0 = _pf_time.monotonic()
        _cand = ecg_clean_worker.process_one(_rel)
        dt = _pf_time.monotonic() - t0
        if _cand.get("status") in ("success", "skipped"):
            _pf_row, _pf_single_secs, _pf_single_rel = _cand, dt, _rel
            break
        print(f"  [WARN] {_rel} -> {_cand.get('status')}: {_cand.get('error','')[:160]}", flush=True)
    assert _pf_row is not None, "first 3 records all failed — inspect §5 data before §8"
    out = OUTPUT_DIR / _pf_row["image_path"]
    assert out.is_file() and out.stat().st_size > 50_000, f"test PNG missing/small: {out}"
    with Image.open(out) as _im:
        assert _im.size == (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT), f"test PNG size {_im.size}"
    est_h = _pf_single_secs * len(records) / max(WORKERS, 1) / 3600
    print(f"  [PASS] single-record render: {_pf_single_rel} -> {out} "
          f"({out.stat().st_size / 2**20:.2f} MiB, {_pf_single_secs:.0f}s; "
          f"upper bound ~{est_h:.1f}h on {WORKERS} workers incl. per-record kit start-up; "
          f"§7b measures the real rate)", flush=True)
    try:
        from IPython.display import display as _display
        _display(Image.open(out))
    except Exception:
        pass
except Exception as exc:
    print(f"  [FAIL] single-record render: {type(exc).__name__}: {exc}", flush=True)
    _pf_failures.append(f"single-record render: {type(exc).__name__}: {exc}")

print(f"Preflight done: {len(records):,} records, "
      f"I0001 {site_counts.get('I0001', 0):,} / I0006 {site_counts.get('I0006', 0):,}.", flush=True)
for _warning in _pf_warnings:
    print("  [WARN]", _warning, flush=True)
if _pf_failures:
    raise RuntimeError("PREFLIGHT FAIL — fix and re-run this cell only:\n- " + "\n- ".join(_pf_failures))
print("PREFLIGHT PASS — run §7b (throughput benchmark) and §7c (time estimate), then §8 (generate).", flush=True)


## 7b. Throughput benchmark (~2-4 min)

Renders one full wave — `WORKERS` batches of `BATCH_SIZE` records, the exact §8 machinery — and reports the real records/s and the ETA for the rest. The images land in `OUTPUT_DIR`, so §8 reuses them as `skipped`; nothing is wasted. The rate should be close to `WORKERS / seconds-per-record`; if it is well below, something other than the CPU is the bottleneck (check `top` for a busy parent process or `iostat` for EBS). Re-running this cell after the wave is rendered reports nothing new, because those records are skipped.


In [ ]:
# §7b: one full wave at the real WORKERS x BATCH_SIZE. Output is kept for §8.
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = str(WORK_DIR) + (
        (os.pathsep + os.environ["PYTHONPATH"]) if os.environ.get("PYTHONPATH") else "")
_bench = records[:WORKERS * BATCH_SIZE]
print(f"Benchmark: {len(_bench):,} records as {WORKERS} batches of {BATCH_SIZE} ...", flush=True)
_t0 = time.monotonic()
with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("spawn"),
                         initializer=ecg_clean_worker.init_worker,
                         initargs=(render_config, str(INPUT_DIR), str(OUTPUT_DIR))) as _pool:
    _futures = [_pool.submit(ecg_clean_worker.process_batch, _bench[i:i + BATCH_SIZE])
                for i in range(0, len(_bench), BATCH_SIZE)]
    _rows = [row for future in _futures for row in future.result()]
_dt = time.monotonic() - _t0
_bc = Counter(row["status"] for row in _rows)
print("Statuses:", dict(_bc))
if _bc["invalid"] or _bc["failed"]:
    for row in _rows:
        if row["status"] in ("invalid", "failed"):
            print("  ", row["source_header"], "->", row["error"][:200])
bench_rate = None
if _bc["success"] < len(_bench) // 2:
    print("Most of this wave was already rendered; no fresh rate to report.")
else:
    _rate = bench_rate = _bc["success"] / _dt
    _left = len(records) - len(_bench)
    print(f"{_rate:.2f} records/s on {WORKERS} workers "
          f"({WORKERS * _dt / _bc['success']:.2f} core-s per record, wave {_dt:.0f}s incl. pool start-up)")
    print(f"ETA for the remaining {_left:,} records: ~{_left / _rate / 3600:.1f} h")


## 7c. Time estimate for the rest of the job (~30 s)

Run after §7b (it uses §7b's measured render rate; without it, it falls back to the pessimistic §7 single-record time). It measures this instance and volume, then prints how long §8 and §9 will take and roughly when the job will finish (UTC):

- **§8 render:** images not yet on disk ÷ §7b records/s.
- **§9 package and upload:** archive size (mean size of the PNGs rendered so far × 113,358, plus metadata) ÷ the slowest stream stage. The stages are (a) the packer thread, which reads PNGs from EBS and hashes them twice, one step after the other, measured by reading the §7b PNGs after dropping them from the page cache; and (b) the S3 upload, measured by uploading a 1 GiB random probe object to `S3_UPLOAD_PREFIX` with the same settings as §9, then deleting it. Set `MEASURE_UPLOAD_MIB = 0` to skip the probe.

This is an estimate: a short EBS read can run faster than a long one, and S3 speed varies. §8's progress lines keep updating the ETA during the run.


In [ ]:
# §7c: estimate the rest of the job from measurements on THIS instance.
#   render (§8)  = images still to render / §7b records-per-second
#   package (§9) = archive bytes / slowest stage of the stream:
#                  packer thread (EBS read + 2 SHA-256 passes, serial) vs S3 upload
import io
from datetime import timedelta

MEASURE_UPLOAD_MIB = 1024  # S3 upload probe size (deleted after); 0 skips it

def _fmt_duration(seconds):
    seconds = max(0, int(round(seconds)))
    h, rem = divmod(seconds, 3600)
    return f"{h}h {rem // 60:02d}m" if h else f"{rem // 60}m {rem % 60:02d}s"

_image_paths = [OUTPUT_DIR / "images" / PurePosixPath(rel).with_suffix(".png") for rel in records]
_done_paths = [p for p in _image_paths if p.is_file()]
_remaining = len(_image_paths) - len(_done_paths)
print(f"Images on disk: {len(_done_paths):,}/{len(_image_paths):,}; still to render: {_remaining:,}")

# --- §8 render rate ---------------------------------------------------------
_rate = globals().get("bench_rate")
if _rate:
    _rate_note = f"§7b measured {_rate:.2f} records/s on {WORKERS} workers"
elif globals().get("_pf_single_secs"):
    _rate = WORKERS / _pf_single_secs
    _rate_note = (f"no §7b rate: preflight single record ({_pf_single_secs:.1f}s incl. kit start-up) "
                  f"x {WORKERS} workers; pessimistic")
else:
    raise RuntimeError("Run §7 and §7b first: there is no measured render rate yet.")
render_s = _remaining / _rate

# --- archive size: measured mean PNG size of the images rendered so far ------
_sample = _done_paths[-2000:]
if not _sample:
    raise RuntimeError("No rendered images yet; run §7b first.")
_mean_png = sum(p.stat().st_size for p in _sample) / len(_sample)
_meta_bytes = sum(p.stat().st_size for p in OUTPUT_DIR.iterdir() if p.is_file())
archive_bytes = _mean_png * len(records) + _meta_bytes + 1024 * len(records)  # + tar headers/padding

# --- §9 stage speeds ----------------------------------------------------------
# EBS read: flush and drop the rendered PNGs from the page cache, then read them
# the way tarfile does, so this is the volume's real small-file read rate.
os.sync()
for p in _sample:
    _fd = os.open(p, os.O_RDONLY)
    try:
        os.posix_fadvise(_fd, 0, 0, os.POSIX_FADV_DONTNEED)
    finally:
        os.close(_fd)
_t0, _read_bytes = time.monotonic(), 0
for p in _sample:
    with p.open("rb") as _fh:
        while _chunk := _fh.read(8 * 2**20):
            _read_bytes += len(_chunk)
read_bps = _read_bytes / max(time.monotonic() - _t0, 1e-6)

_buf = os.urandom(256 * 2**20)
_t0 = time.monotonic()
hashlib.sha256(_buf).digest()
hash_bps = len(_buf) / max(time.monotonic() - _t0, 1e-6)
packer_bps = 1 / (1 / read_bps + 2 / hash_bps)  # same thread: read, then whole + part hashes

upload_bps = None
if MEASURE_UPLOAD_MIB:
    _up = urlparse(S3_UPLOAD_PREFIX)
    _probe_key = "/".join(p for p in (_up.path.strip("/"), ".throughput_probe.bin") if p)
    _probe = io.BytesIO(os.urandom(MEASURE_UPLOAD_MIB * 2**20))
    _probe_cfg = TransferConfig(multipart_threshold=64 * 2**20, multipart_chunksize=64 * 2**20,
                                max_concurrency=32, use_threads=True,
                                preferred_transfer_client="classic")
    try:
        _t0 = time.monotonic()
        s3.upload_fileobj(_probe, _up.netloc, _probe_key,
                          ExtraArgs={"ChecksumAlgorithm": "SHA256"}, Config=_probe_cfg)
        upload_bps = MEASURE_UPLOAD_MIB * 2**20 / max(time.monotonic() - _t0, 1e-6)
    finally:
        try:
            s3.delete_object(Bucket=_up.netloc, Key=_probe_key)
        except Exception:
            pass
    del _probe

stream_bps = min(packer_bps, upload_bps) if upload_bps else packer_bps
package_s = archive_bytes / stream_bps
_bottleneck = ("S3 upload" if upload_bps and upload_bps < packer_bps
               else "EBS read" if read_bps < hash_bps / 2 else "SHA-256 hashing")

total_s = render_s + package_s
_now = datetime.now(timezone.utc)
_mib = lambda bps: f"{bps / 2**20:,.0f} MiB/s"
print()
print(f"§8 render   : {_fmt_duration(render_s):>8}  ({_remaining:,} images; {_rate_note})")
print(f"§9 package  : {_fmt_duration(package_s):>8}  ({archive_bytes / 2**30:,.1f} GiB archive at ~{_mib(stream_bps)}; "
      f"bottleneck: {_bottleneck})")
print(f"             EBS read {_mib(read_bps)} | SHA-256 {_mib(hash_bps)} | "
      f"S3 upload {_mib(upload_bps) if upload_bps else 'not measured'}")
print(f"Total       : {_fmt_duration(total_s):>8}  -> done around "
      f"{(_now + timedelta(seconds=total_s)):%Y-%m-%d %H:%M} UTC")
if read_bps < 300 * 2**20:
    _fast = archive_bytes / min(1 / (1 / (1000 * 2**20) + 2 / hash_bps), upload_bps or float("inf"))
    print(f"Tip: EBS reads at {_mib(read_bps)}. Raising the gp3 volume to 1,000 MB/s "
          f"(EC2 console -> Volumes -> Modify, applies live) would cut §9 to ~{_fmt_duration(_fast)}.")


## 8. Generate images and reports

At most `MAX_PENDING_BATCHES` are queued, so neither the task queue nor results grow with the full dataset. Status CSVs are written by the parent only; workers write disjoint image paths. If some records are corrupt, the job continues and their paths/reasons are recorded. `processed = successful + skipped + invalid + failed` in the final summary. `successful` counts newly written images, while `skipped` counts valid images reused from an earlier run.

Only run after the §7 preflight cell prints `PREFLIGHT PASS`.

Every progress line ends with `ETA`, the time left at the rate of newly rendered images so far (reused images are not counted, so a resumed run is not over-optimistic).


In [ ]:
# Spawn-context children inherit os.environ, not sys.path: re-ensure WORK_DIR
# is on PYTHONPATH (normally set in §6) so children can import ecg_clean_worker.
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = str(WORK_DIR) + (
        (os.pathsep + os.environ["PYTHONPATH"]) if os.environ.get("PYTHONPATH") else "")

fields = ["source_header", "image_path", "status", "error", "sample_rate_hz",
          "samples", "duration_s", "image_width", "image_height", "image_bytes"]
manifest_path = OUTPUT_DIR / "manifest.csv"
failures_path = OUTPUT_DIR / "failures.csv"
counts = Counter()
image_bytes_total = 0
started = time.monotonic()
last_report = 0

def batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

def record_result(row, manifest_writer, failure_writer):
    global image_bytes_total
    counts[row["status"]] += 1
    image_bytes_total += int(row["image_bytes"] or 0)
    manifest_writer.writerow(row)
    if row["status"] in {"invalid", "failed"}:
        failure_writer.writerow(row)

with manifest_path.open("w", newline="") as manifest_file, failures_path.open("w", newline="") as failure_file:
    manifest_writer = csv.DictWriter(manifest_file, fieldnames=fields)
    failure_writer = csv.DictWriter(failure_file, fieldnames=fields)
    manifest_writer.writeheader()
    failure_writer.writeheader()

    batch_iter = iter(batches(records, BATCH_SIZE))
    context = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=context,
                             initializer=ecg_clean_worker.init_worker,
                             initargs=(render_config, str(INPUT_DIR), str(OUTPUT_DIR))) as pool:
        pending = {}
        for _ in range(MAX_PENDING_BATCHES):
            batch = next(batch_iter, None)
            if batch is None:
                break
            pending[pool.submit(ecg_clean_worker.process_batch, batch)] = batch
        while pending:
            done, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in done:
                batch = pending.pop(future)
                try:
                    result_rows = future.result()
                except Exception as exc:
                    result_rows = [{"source_header": rel,
                                    "image_path": (Path("images") / Path(rel).with_suffix(".png")).as_posix(),
                                    "status": "failed", "error": f"Worker batch failure: {type(exc).__name__}: {exc}"[:1000],
                                    "sample_rate_hz": "", "samples": "", "duration_s": "",
                                    "image_width": "", "image_height": "", "image_bytes": ""}
                                   for rel in batch]
                for row in result_rows:
                    record_result(row, manifest_writer, failure_writer)
                next_batch = next(batch_iter, None)
                if next_batch is not None:
                    pending[pool.submit(ecg_clean_worker.process_batch, next_batch)] = next_batch
            processed = sum(counts.values())
            if processed - last_report >= PROGRESS_EVERY or processed == len(records):
                elapsed = max(time.monotonic() - started, 1)
                new_rate = counts["success"] / elapsed
                eta = (f"{(len(records) - processed) / new_rate / 3600:.1f} h"
                       if new_rate > 0 else "n/a")
                print(f"{processed:,}/{len(records):,} ({processed / elapsed:.1f}/s, "
                      f"new {new_rate:.2f}/s) | new {counts['success']:,}, reused {counts['skipped']:,}, "
                      f"invalid {counts['invalid']:,}, failed {counts['failed']:,} | ETA {eta}", flush=True)
                manifest_file.flush()
                failure_file.flush()
                last_report = processed

summary = {
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "source": S3_SOURCE,
    "discovered": len(records),
    "processed": sum(counts.values()),
    "successful": counts["success"],
    "skipped": counts["skipped"],
    "invalid": counts["invalid"],
    "failed": counts["invalid"] + counts["failed"],
    "runtime_failures": counts["failed"],
    "image_bytes": image_bytes_total,
    "elapsed_seconds": round(time.monotonic() - started, 1),
    "render_config_sha256": config_hash,
}
if summary["processed"] != len(records):
    raise RuntimeError("Generation did not account for every discovered header")
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
print("Manifest:", manifest_path, "| failures:", failures_path)


## 9. Package, hash, and upload in one pass

No archive is written to local disk. A packer thread writes the TAR into an in-memory pipe while boto3 uploads it to S3 as 64 MiB multipart parts (32 in parallel), and every byte is SHA-256-hashed as it passes through. The images are only read, never modified; the archive members are byte-identical copies of the PNGs on disk. Archive contents are unchanged from before: `images/I0001/...` and `images/I0006/...`, plus the source record manifest, selection report, final counts, documentation, rendering manifest, failure log, configuration, and summary (stray `*.part` temp files from a killed worker are left out).

Integrity checks: each part is uploaded with a SHA-256 that S3 validates on receipt, and after completion the object's size and S3's composite SHA-256 are checked against the bytes this notebook streamed. The image count is also checked. If any check fails, the uploaded object is deleted and the cell raises. If packing or the upload fails part-way, boto3 aborts the multipart upload, so no partial object is left behind. To retry, re-run this cell; it streams again from the PNGs on disk. The `<archive>.sha256` sidecar (same `sha256sum` format as the source and as `clean_ecg_signal_images_kit_v1.tar.sha256`) is uploaded next to the archive, and a copy is kept in `WORK_DIR`. Plain TAR is the default (PNG is already compressed); `ARCHIVE_FORMAT = "tar.gz"` also streams, but single-threaded gzip makes it much slower. Review `summary.json` and `failures.csv` before using any images for training.


In [ ]:
# §9: pack, hash, and upload in ONE pass. tarfile writes into an in-memory pipe
# (packer thread) while boto3 uploads 64 MiB multipart parts from it (this
# thread); no archive is ever written to local disk. Images are only read.
import base64
import io
import threading
from collections import deque
from botocore.config import Config as BotoConfig

STREAM_PART_BYTES = 64 * 2**20            # 105 GiB -> ~1,700 parts (S3 max 10,000)
STREAM_BUFFER_BYTES = 4 * STREAM_PART_BYTES

class TarUploadStream(io.RawIOBase):
    """Pipe between tarfile (writer thread) and the S3 multipart upload (reader).

    Every byte is hashed once, in order, as it is written: the whole-archive
    SHA-256 for the sidecar, and per-part SHA-256s to check S3's composite
    checksum of the assembled object. read(n) blocks until n bytes or end of
    stream, so every S3 part except the last is exactly n bytes.
    """
    def __init__(self, part_bytes, max_buffered):
        super().__init__()
        assert max_buffered > part_bytes
        self.part_bytes, self.max_buffered = part_bytes, max_buffered
        self.digest = hashlib.sha256()
        self.part_digests = []
        self.bytes_written = 0
        self._part, self._part_fill = hashlib.sha256(), 0
        self._chunks, self._buffered = deque(), 0
        self._eof, self._error, self._aborted = False, None, False
        self._cv = threading.Condition()

    def readable(self):
        return True

    def writable(self):
        return True

    def tell(self):
        return self.bytes_written

    def _hash(self, data):
        self.digest.update(data)
        self.bytes_written += len(data)
        view = memoryview(data)
        while view:
            take = min(len(view), self.part_bytes - self._part_fill)
            self._part.update(view[:take])
            self._part_fill += take
            view = view[take:]
            if self._part_fill == self.part_bytes:
                self.part_digests.append(self._part.digest())
                self._part, self._part_fill = hashlib.sha256(), 0

    def write(self, data):
        data = bytes(data)
        with self._cv:
            while self._buffered >= self.max_buffered and not self._aborted:
                self._cv.wait()
            if self._aborted:
                raise IOError("S3 upload stopped; archive stream aborted")
        self._hash(data)  # single writer thread: hashing order == stream order
        with self._cv:
            self._chunks.append(data)
            self._buffered += len(data)
            self._cv.notify_all()
        return len(data)

    def read(self, size=-1):
        with self._cv:
            while not self._eof and (size is None or size < 0 or self._buffered < size):
                self._cv.wait()
            if self._error is not None:
                raise IOError(f"archive packing failed: {type(self._error).__name__}: {self._error}")
            want = self._buffered if size is None or size < 0 else min(size, self._buffered)
            out = bytearray()
            while len(out) < want:
                chunk = self._chunks.popleft()
                need = want - len(out)
                if len(chunk) > need:
                    self._chunks.appendleft(chunk[need:])
                    chunk = chunk[:need]
                out += chunk
            self._buffered -= len(out)
            self._cv.notify_all()
            return bytes(out)

    def finish(self, error=None):
        with self._cv:
            if self._part_fill:
                self.part_digests.append(self._part.digest())
                self._part_fill = 0
            self._eof, self._error = True, error
            self._cv.notify_all()

    def abort(self):
        with self._cv:
            self._aborted = True
            self._cv.notify_all()

    def s3_composite_sha256(self):
        """S3's COMPOSITE ChecksumSHA256 for these bytes uploaded as parts."""
        if len(self.part_digests) <= 1:
            return base64.b64encode(self.digest.digest()).decode()  # single PutObject
        combined = hashlib.sha256(b"".join(self.part_digests)).digest()
        return f"{base64.b64encode(combined).decode()}-{len(self.part_digests)}"

upload_prefix = urlparse(S3_UPLOAD_PREFIX)
if upload_prefix.scheme != "s3" or not upload_prefix.netloc:
    raise ValueError("S3_UPLOAD_PREFIX must be s3://bucket/ or s3://bucket/prefix/")
archive_name = f"{ARCHIVE_BASENAME}.{ARCHIVE_FORMAT}"
upload_bucket = upload_prefix.netloc
upload_key = "/".join(part for part in (upload_prefix.path.strip("/"), archive_name) if part)
uploaded_uri = f"s3://{upload_bucket}/{upload_key}"
expected_images = summary["successful"] + summary["skipped"]
mode = "w" if ARCHIVE_FORMAT == "tar" else "w:gz"
image_prefix = OUTPUT_DIR.name + "/images/"
packed = Counter()

def _member_filter(info):
    if info.name.endswith(".part"):  # temp file of a killed worker: never package
        packed["skipped_part_files"] += 1
        return None
    if info.isfile() and info.name.startswith(image_prefix) and info.name.endswith(".png"):
        packed["images"] += 1
    return info

def _pack(stream):
    try:
        with tarfile.open(fileobj=stream, mode=mode, copybufsize=8 * 2**20,
                          **({"compresslevel": 1} if mode == "w:gz" else {})) as archive:
            archive.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name, recursive=True, filter=_member_filter)
        stream.finish()
    except BaseException as exc:
        stream.finish(exc)

upload_client = aws_session.client("s3", config=BotoConfig(
    retries={"max_attempts": 10, "mode": "standard"}, max_pool_connections=64))
stream_transfer = TransferConfig(multipart_threshold=STREAM_PART_BYTES,
                                 multipart_chunksize=STREAM_PART_BYTES,
                                 max_concurrency=32, use_threads=True,
                                 preferred_transfer_client="classic")
stream_transfer.max_in_memory_upload_chunks = 32  # ~2 GiB of parts in flight

stream = TarUploadStream(STREAM_PART_BYTES, STREAM_BUFFER_BYTES)
packer = threading.Thread(target=_pack, args=(stream,), name="tar-packer", daemon=True)
print(f"Streaming {OUTPUT_DIR.name} -> {uploaded_uri} (tar + SHA-256 + upload in one pass) ...", flush=True)
_t0 = time.monotonic()
packer.start()
try:
    # SHA256 per part: S3 rejects any part whose bytes do not match, and the
    # finished object carries a composite checksum we verify below.
    upload_client.upload_fileobj(stream, upload_bucket, upload_key,
                                 ExtraArgs={"ChecksumAlgorithm": "SHA256"}, Config=stream_transfer)
except BaseException:
    stream.abort()      # unblock the packer; boto3 has already aborted the multipart upload
    packer.join()
    raise
packer.join()
elapsed = time.monotonic() - _t0
archive_sha256 = stream.digest.hexdigest()

remote = upload_client.head_object(Bucket=upload_bucket, Key=upload_key, ChecksumMode="ENABLED")
problems = []
if int(remote["ContentLength"]) != stream.bytes_written:
    problems.append(f"S3 size {int(remote['ContentLength']):,} != streamed {stream.bytes_written:,} bytes")
if remote.get("ChecksumSHA256") != stream.s3_composite_sha256():
    problems.append(f"S3 ChecksumSHA256 {remote.get('ChecksumSHA256')!r} != local {stream.s3_composite_sha256()!r}")
if packed["images"] != expected_images:
    problems.append(f"archive has {packed['images']:,} images; expected {expected_images:,}")
if problems:
    upload_client.delete_object(Bucket=upload_bucket, Key=upload_key)
    raise RuntimeError("Uploaded archive failed verification and was deleted:\n- " + "\n- ".join(problems))

sidecar_body = f"{archive_sha256}  {archive_name}\n".encode()
(WORK_DIR / (archive_name + ".sha256")).write_bytes(sidecar_body)
upload_client.put_object(Bucket=upload_bucket, Key=upload_key + ".sha256", Body=sidecar_body)
print(f"Uploaded {packed['images']:,} images; {stream.bytes_written / 2**30:.2f} GiB in "
      f"{len(stream.part_digests):,} parts, {elapsed / 60:.1f} min "
      f"({stream.bytes_written / max(elapsed, 1e-9) / 2**20:.0f} MiB/s) -> {uploaded_uri}")
print(f"S3 composite checksum verified: {remote['ChecksumSHA256']}")
print(f"SHA-256 {archive_sha256} -> {uploaded_uri}.sha256")
if packed["skipped_part_files"]:
    print(f"Left out {packed['skipped_part_files']} stray *.part temp files")
print("Final status:", "complete" if summary["failed"] == 0 else "complete with record failures; inspect failures.csv")


## Run verification

After the last cell, confirm `summary.json` has `processed == discovered` (113,358), inspect `failed` and `failures.csv`, and confirm the packaged image count equals `successful + skipped`. `record_manifest.csv` in the image archive is the clean12 source manifest: join on `record_path` (rendering manifest column `source_header`) to get labels and `patient_key` for each PNG. Check the S3 URI, byte count, composite checksum, and SHA-256 printed after upload. To check a downloaded copy later, run `sha256sum -c clean_ecg_signal_images_clean12.tar.sha256`. Rerunning the notebook reuses the SHA-256 verified source archive, extracted tree, and valid PNGs. If you change rendering settings, choose a new output directory and archive basename so different renderings cannot mix.

## Storage planning

Measured: the clean12 source archive is **5.48 GiB** (5,883,697,122 bytes) and holds **113,358 records** (74,356 I0001 `.mat` + 39,002 I0006 `.dat`, all native 500 Hz). The extracted records take **13.5 GiB**. Rendered 1024x768 PNGs average **0.942 MiB** (0.92-1.05 MiB over 24 clean12 renders), so the image set is about **104 GiB**. Because §9 streams the TAR to S3, no local archive is needed.

| On the EBS volume at peak (end of §8 / during §9) | GiB |
|---|---|
| Source archive (kept so reruns skip the download) | 5.5 |
| Extracted records | 13.5 |
| Rendered images (113,358 x 0.942 MiB) | 104 |
| Manifests, logs, kit, worker | < 1 |
| OS + Python packages (if on the same volume) | ~8-10 |
| **Total** | **~132** |

A **150 GB gp3** volume gives ~15% headroom (160 GB if you want more margin). You can delete `input/ECG_500Hz_clean12.tar.gz` after §3 to free 5.5 GiB, at the cost of re-downloading on a rerun. RAM: render ~7 GiB (16 kit processes at ~240 MB plus workers) plus `/dev/shm` scratch < 1 GiB; §9 holds ~2.3 GiB of parts in flight. Both fit easily in the c8a.4xlarge's 32 GiB.

## c8a.4xlarge tuning

Per record, ~90% of the time is the kit's matplotlib drawing (tick/grid construction, line drawing, Agg rasterization, plus coordinate bookkeeping the kit computes but this pipeline discards), all single-threaded Python inside one kit process; staging, recolour, resize, and PNG save are the rest. The work is therefore one CPU core per worker and records are independent:

- `WORKERS = os.cpu_count()` = **16** (was 14). c8a vCPUs are physical cores (no SMT), and the parent process is idle apart from CSV writes. Expect ~14% more throughput than 14 workers.
- `BATCH_SIZE = 48` (was 24): the kit's 1-3 s import is paid once per batch; the end-of-run tail is at most one batch (~2 min).
- Kit scratch in `/dev/shm`; recolour through uint8 point tables (bit-identical, ~80 ms less per record).
- Memory: a kit process peaks at ~240 MB RSS, so 16 workers need ~6-7 GiB of the 32 GiB.
- At the kit-v1 c8a pace (~2.6 core-seconds per record), the render is about **113,358 x 2.6 / 16 ~= 5.1 h**. §7b measures the real number before you commit.
- **§9 streaming:** the images are read from EBS exactly once and nothing is written. At the gp3 default of 125 MB/s that read takes ~15 min; raising the volume to **500-1,000 MB/s and 6,000 IOPS** (EC2 console → Volumes → Modify; applies live, costs cents per hour) brings §9 down to a few minutes, after which hashing and the S3 upload are the limit. Building a local TAR, hashing it, and then uploading it would read ~315 GiB and write ~105 GiB, and need ~105 GiB more disk.
